## Cell 1 — Imports and configuration

In [ ]:
# Cell 1: imports and the whole configuration for this notebook.
#
# Pipeline, in the order the cells below implement it:
#   paper text taken from the validation workbook                 (Cell 11)
#   -> characterization / testing sections cut off                (Cell 3)
#   -> sentences that carry an operation cue AND a value          (Cells 4-6)
#   -> each kept sentence parsed by the trained Snowball2 parsers (Cells 7-9)
#   -> the vessel of each step read, sealed-vessel rule R-T3      (Cells 4, 10)
#   -> OperationTime records bound to OperationTemperature rows   (Cell 12)
#   -> heating-rate false positives dropped                       (Cell 13)
#   -> each heating rate bound to the set point it ramps to       (Cells 13-14)
#   -> one row per operation event                                (Cells 14-16)
#
# The corpus is read from the validation workbook itself, so the text that is
# parsed and the text that was annotated are the same characters by construction.
# Nothing downstream matches one text source against another.

import re
import json
from pathlib import Path

import pandas as pd

from chemdataextractor.doc.document import Document
from chemdataextractor.doc.text import Paragraph
from chemdataextractor.relex.snowball import Snowball

from operation_model import OperationTemperature, OperationTime


# Corpus. Text, title, ID and year all come from this one sheet.
VALIDATION_FILE = "0_Annotated_12_Years_Fixed_Sep_24_2026.xlsx"
VALIDATION_SHEET = "0_Validation_Set"
TEXT_COLUMN = "Preparation_Text"
SAVE_ROOT_DIR = "operation_results_ALL"

# Snowball2 phrase-list checkpoints written by notebook 001.
OPERATION_TEMPERATURE_CHECKPOINT = Path("OperationTemperature_list.pkl")
OPERATION_TIME_CHECKPOINT = Path("OperationTime_list.pkl")

# Runtime Snowball settings.
RUNTIME_SAVE_DIR = Path("Snowball_runtime")
RUNTIME_TEMPERATURE_FILE_NAME = "OperationTemperature_runtime"
RUNTIME_TIME_FILE_NAME = "OperationTime_runtime"
CONFIDENCE_LIMIT = 1.0

# Method switches. These are the ones that change what is reported.
ENABLE_SECTION_FILTER = True             # cut characterization / testing sections
ENABLE_TIME_SNOWBALL_MODEL = True        # run OperationTime as well as OperationTemperature
DROP_MODEL_TEMPERATURE_RATES = True      # drop rows whose temperature is a ramp rate
KEEP_MODEL_TIME_ONLY_ROWS = True         # "stirred for 3 h" is a result, not a discard
COLLAPSE_MATERIAL_DUPLICATE_ROWS = True  # one row per event, materials joined into a list
RESTRICT_TO_IN_SCOPE_OPERATIONS = False  # keep out-of-scope operations, tagged not dropped
ATTACH_HEATING_RATES = True              # bind each heating rate to the set point it ramps to
ENABLE_SEALED_VESSEL_RULE = True         # rule R-T3: generic thermal step in a sealed vessel

# The nine operations the parser was trained on. Rows outside this set are kept
# and flagged with operation_in_training_scope so they can be separated later.
IN_SCOPE_OPERATIONS = {
    "calcination",
    "drying",
    "stirring",
    "heating",
    "aging",
    "sonication",
    "thermal hold",
    "hydrothermal treatment",
    "evaporation",
}

# Rule R-T3 (Rule_RT3_Sealed_Vessel.md). A vessel is sealed when its text names an
# autoclave or a Teflon/PTFE liner, or the clause says sealed. The rule only relabels a
# step whose verb names no specific operation (heated, kept, maintained, held, treated);
# a specific verb (aged, dried, calcined, stirred) keeps its own label whatever the vessel.
SEALED_VESSEL_RE = re.compile(r"autoclave|teflon|ptfe|\bsealed\b", flags=re.I)
SEALED_VESSEL_RULE_OPERATIONS = {"heating", "thermal hold", "thermal treatment"}
SEALED_VESSEL_OPERATION = "hydrothermal treatment"

OPERATION_MODELS = (
    [OperationTemperature, OperationTime]
    if ENABLE_TIME_SNOWBALL_MODEL
    else [OperationTemperature]
)

# Diagnostics. Nothing downstream reads these workbooks.
SAVE_OPERATION_SENTENCE_AUDIT = True
SAVE_MODEL_TIME_AUDIT = True
RUN_VERBOSE = False

## Cell 2 — Text normalization

In [ ]:
# Cell 2: one normalization pass, plus the surface patterns for temperature and time.
#
# Every stage below reads normalized text, so a unit written as a degree sign, as
# "deg C" or glued to its number is the same string everywhere.

NUM_RE_PART = r"[+\-\u2212]?(?:\d+(?:[.,]\d+)?|\.\d+)"
TEMP_RANGE_SEP_RE_PART = r"(?:-|−|–|—|to|~|〜|～)"

CELSIUS_UNIT_RE_PART = (
    r"(?:"
    r"°\s*C|"
    r"º\s*C|"
    r"˚\s*C|"
    r"⁰\s*C|"
    r"℃|"
    r"deg(?:ree(?:s)?)?\.?\s*C(?:elsius)?|"
    r"degrees?\s+C(?:elsius)?|"
    r"celsius|"
    r"centigrade"
    r")"
)

FAHRENHEIT_UNIT_RE_PART = (
    r"(?:"
    r"°\s*F|"
    r"º\s*F|"
    r"˚\s*F|"
    r"⁰\s*F|"
    r"℉|"
    r"deg(?:ree(?:s)?)?\.?\s*F(?:ahrenheit)?|"
    r"degrees?\s+F(?:ahrenheit)?|"
    r"fahrenheit"
    r")"
)

KELVIN_UNIT_RE_PART = r"(?:K|kelvin)"

TEMP_UNIT_RE_PART = (
    rf"(?:{CELSIUS_UNIT_RE_PART}|{FAHRENHEIT_UNIT_RE_PART}|{KELVIN_UNIT_RE_PART})"
)

TEMP_RE = re.compile(
    rf"""
    (?<![\w/])
    (?P<value>
        {NUM_RE_PART}
        (?:
            \s*{TEMP_RANGE_SEP_RE_PART}\s*
            {NUM_RE_PART}
        )?
    )
    \s*
    (?P<unit>{TEMP_UNIT_RE_PART})
    (?![A-Za-z0-9])
    """,
    flags=re.I | re.VERBOSE,
)

# A degree sign with no unit letter. Only trusted inside an operation clause.
BARE_DEGREE_RE = re.compile(
    rf"""
    (?<![\w/])
    (?P<value>{NUM_RE_PART})
    \s*[°º˚⁰]
    (?!\s*(?:[CFK]\b|celsius|fahrenheit|kelvin))
    """,
    flags=re.I | re.VERBOSE,
)

TIME_RE = re.compile(
    r"(?:(?P<value>[-+]?\d+(?:\.\d+)?(?:\s*(?:-|–|—|to)\s*[-+]?\d+(?:\.\d+)?)?)"
    r"\s*"
    r"(?P<unit>h|hr|hrs|hour|hours|min|mins|minute|minutes|s|sec|secs|second|seconds|d|day|days)\b"
    r"|(?P<surface_only>overnight)\b)",
    flags=re.I,
)

# A value written as a range is not a valid target under the annotation policy.
TEMP_RANGE_VALUE_RE = re.compile(
    rf"^\s*{NUM_RE_PART}\s*{TEMP_RANGE_SEP_RE_PART}\s*{NUM_RE_PART}\s*$",
    flags=re.I,
)


def normalize_text_for_operation_snowball(text):
    if not isinstance(text, str):
        return text

    # Zero-width format characters (publisher HTML inserts U+200B between a
    # number and its unit) are invisible but break every value regex below.
    text = re.sub("[\u200b\u200c\u200d\u2060\ufeff\u00ad]", "", text)

    text = (
        text.replace(" ", " ")
        .replace(" ", " ")
        .replace(" ", " ")
        .replace(" ", " ")
        .replace("‑", "-")
        .replace("−", "-")
    )

    text = (
        text.replace("℃", "°C")
        .replace("℉", "°F")
        .replace("º", "°")
        .replace("⁰", "°")
        .replace("˚", "°")
    )

    # 500 deg C / 500 degrees Celsius -> 500 °C
    text = re.sub(
        rf"(?<=\d)\s*(?:{CELSIUS_UNIT_RE_PART})(?![A-Za-z0-9])",
        " °C",
        text,
        flags=re.I,
    )

    # 500°C / 500 ° C -> 500 °C
    text = re.sub(
        r"(?<=\d)\s*°\s*([CF])\b",
        lambda m: " °" + m.group(1).upper(),
        text,
    )

    # 3h / 10mins -> 3 h / 10 mins
    text = re.sub(
        r"(?<=\d)\s*(h|hr|hrs|hour|hours|min|mins|minute|minutes|s|sec|secs|second|seconds)\b",
        r" \1",
        text,
        flags=re.I,
    )

    return re.sub(r"\s+", " ", text).strip()


def normalize_temperature_unit(unit):
    if unit is None:
        return None

    unit_text = str(unit).strip().casefold()
    unit_text = unit_text.replace("º", "°").replace("˚", "°").replace("⁰", "°")
    unit_text = re.sub(r"[\s.]+", "", unit_text)

    if unit_text in {
        "℃", "°c", "degc", "degreec", "degreesc",
        "degreecelsius", "degreescelsius", "celsius", "centigrade",
    }:
        return "°C"

    if unit_text in {
        "℉", "°f", "degf", "degreef", "degreesf",
        "degreefahrenheit", "degreesfahrenheit", "fahrenheit",
    }:
        return "°F"

    if unit_text in {"k", "kelvin"}:
        return "K"

    return str(unit).strip()


def canonical_temperature_value(value):
    """Reduce a value surface to the number or numbers it contains."""
    if value is None:
        return None

    value_text = str(value).strip().replace("–", "-").replace("—", "-")
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", value_text)

    if not numbers:
        return value_text

    return numbers[0] if len(numbers) == 1 else " to ".join(numbers)

## Cell 3 — Characterization section filter

In [ ]:
# Cell 3: cut a paper at the first characterization or testing section heading, so
# instrument settings and reactor test conditions are never read as synthesis
# parameters.
#
# The cut is made only at a line that looks like a heading, or at a heading the text
# export ran onto the end of the paragraph before it (". Catalysts characterization"
# closing a long line). Two cases are deliberately not cut, because cutting at either deletes the whole procedure: a
# heading that also announces preparation, since the procedure follows it; and a
# match inside a prose sentence, since that is not a section break.

STOP_TERM_RE = re.compile(
    r"(?:"
    r"materials?\s+characteri[sz]ations?|"
    r"catalysts?\s+characteri[sz]ations?|"
    r"characteri[sz]ations?(?:\s+of\b|\s*$|\s*:)|"
    r"catalytic\s+activity(?:\s+evaluation|\s+test(?:s|ing)?|\s+measurements?)?|"
    r"activity\s+(?:test|tests|testing|evaluation|measurements?)|"
    r"performance\s+(?:test|tests|testing|evaluation|measurements?)|"
    r"reaction\s+(?:test|tests|testing|evaluation)|"
    r"catalyst\s+(?:testing|tests?|evaluation)|"
    r"catalysts?\s+evaluations?|"
    r"evaluations?\s+of\s+(?:the\s+)?catalysts?|"
    r"catalytic\s+(?:test|tests|testing|evaluation|performance|measurements?)"
    r")",
    flags=re.I,
)

PREP_VETO_RE = re.compile(
    r"\b(?:preparation|preparing|synthesis|synthesi[sz]ed|synthesi[sz]ation|"
    r"chemicals|reagents|fabrication|experimental)\b",
    flags=re.I,
)

SECTION_NUMBER_RE = re.compile(r"^\s*\d+(?:\.\d+){0,3}[.)]?\s*$")

# Headings in this corpus are short standalone lines; body text runs much longer.
HEADING_MAX_CHARS = 90

# A heading run onto a prose line: it follows a sentence end, may carry a section
# number, starts with a capital and finishes the line without sentence punctuation.
INLINE_HEADING_RE = re.compile(
    r"[.;]\s+(?P<heading>(?:\d+(?:\.\d+){0,3}\.?\s+)?[A-Z][^.;\n]{0,%d})\s*$"
    % HEADING_MAX_CHARS
)


def is_heading_line(line):
    """True if the line looks like a section heading rather than prose."""
    stripped = line.strip()

    if not stripped or len(stripped) > HEADING_MAX_CHARS:
        return False

    # Prose ends in sentence punctuation; a numbered heading line does not.
    if stripped.endswith((".", ";")) and not SECTION_NUMBER_RE.match(stripped):
        return False

    return True


def is_stop_heading(heading):
    return bool(STOP_TERM_RE.search(heading)) and not PREP_VETO_RE.search(heading)


def filter_text_after_stop_heading(text):
    """Return (kept_text, removed_heading). The heading is kept for the audit."""
    if not isinstance(text, str) or not text.strip():
        return text, None

    offset = 0

    for line in text.splitlines(keepends=True):
        cut_at, heading = None, None

        if is_heading_line(line):
            if is_stop_heading(line):
                cut_at, heading = offset, line.strip()
        else:
            inline = INLINE_HEADING_RE.search(line)

            if inline and is_stop_heading(inline.group("heading")):
                cut_at, heading = offset + inline.start("heading"), inline.group("heading").strip()

        if cut_at is not None:
            kept = text[:cut_at].strip()
            return (kept, heading) if kept else (text, None)

        offset += len(line)

    return text, None

## Cell 4 — Operation vocabulary and clause splitting

In [ ]:
# Cell 4: build the operation vocabulary from the annotations, not by hand.
#
# One trigger term is often annotated with several operations: "kept" appears as
# thermal hold (8), resting (2), stirring (2), heating (2), drying (1) and
# evaporation (1). Each term is therefore resolved once, to the operation it
# carries most often in the training data, with ties broken by that operation's
# overall frequency. Nothing is added by hand. A term whose support is genuinely
# tied and thin is fixed by annotating a more specific trigger span, not in code.

from collections import Counter

OPERATION_TERMS_CSV = Path("positive_operation_events.csv")


def operation_term_pattern(term):
    tokens = [token for token in re.split(r"[\s\-]+", str(term).strip()) if token]

    if not tokens:
        return None

    return r"[-\s]+".join(re.escape(token) for token in tokens)


def load_training_operation_patterns(csv_path=OPERATION_TERMS_CSV):
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing Step 1 operation-event table: {csv_path}")

    operation_df = pd.read_csv(csv_path, dtype=str).fillna("")

    required_columns = {"operation_type", "operation_verb_text_span"}
    missing_columns = required_columns - set(operation_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns in {csv_path}: {sorted(missing_columns)}"
        )

    term_operation_support = Counter()
    operation_support = Counter()

    for _, row in operation_df.iterrows():
        operation_type = str(row["operation_type"]).strip().lower()
        operation_term = str(row["operation_verb_text_span"]).strip().lower()

        if not operation_type or not operation_term:
            continue

        term_operation_support[(operation_term, operation_type)] += 1
        operation_support[operation_type] += 1

    best_for_term = {}

    for (term_key, operation_type), support in term_operation_support.items():
        candidate = (support, operation_support[operation_type], operation_type)

        if best_for_term.get(term_key) is None or candidate > best_for_term[term_key]:
            best_for_term[term_key] = candidate

    patterns = []
    seen = set()

    for _, row in operation_df.iterrows():
        operation_term = str(row["operation_verb_text_span"]).strip()
        term_key = operation_term.lower()

        if not term_key or term_key in seen:
            continue

        pattern = operation_term_pattern(operation_term)

        if pattern is None:
            continue

        patterns.append({
            "operation_type": best_for_term[term_key][2],
            "operation_term": operation_term,
            "pattern": pattern,
        })
        seen.add(term_key)

    if not patterns:
        raise ValueError("No operation patterns were loaded from the operation-event table.")

    # Longest term first, so a multi-word trigger wins over a substring of itself.
    return sorted(
        patterns,
        key=lambda item: (-len(item["operation_term"]), item["operation_term"].lower()),
    )


TRAINED_OPERATION_PATTERNS = load_training_operation_patterns()

OPERATION_WORD_RE = "|".join(item["pattern"] for item in TRAINED_OPERATION_PATTERNS)

OPERATION_CUE_RE = re.compile(r"\b(?:" + OPERATION_WORD_RE + r")\b", flags=re.I)

EVENT_PATTERNS = [
    (item["operation_type"], re.compile(r"\b(?:" + item["pattern"] + r")\b", flags=re.I))
    for item in TRAINED_OPERATION_PATTERNS
]


def normalize_specifier(specifier):
    """Map a raw trigger word onto a canonical operation name, e.g. 'calcined' -> 'calcination'."""
    if specifier is None:
        return None

    text = str(specifier).strip()

    if not text or text.lower() == "nan":
        return None

    for event_name, pattern in EVENT_PATTERNS:
        if pattern.search(text):
            return event_name

    return text.lower()


# Clause boundaries. Used only to decide whether a bare degree sign sits inside an
# operation clause; the parser itself still receives whole sentences.
CLAUSE_SPLIT_RE = re.compile(
    r"(?<=[.!?])\s+|"
    r";\s+|"
    r"\s*,\s*(?=(?:then|followed by|subsequently|afterward|afterwards|finally|thereafter|next)\b)|"
    r"\s*,\s*(?=(?:the\s+|this\s+|it\s+|which\s+)?"
    r"(?:resulting\s+)?(?:sample|slurry|mixture|solid|powder|solution|precipitate|product|material|catalyst)?"
    r"\s*(?:was|were|is|are)\s+(?:then\s+|subsequently\s+|finally\s+)?(?:"
    + OPERATION_WORD_RE +
    r")\b)|"
    r"\s+\b(?:then|subsequently|afterward|afterwards|finally|thereafter|next)\b\s+|"
    r"\s+\band\s+(?=(?:was\s+|were\s+|is\s+|are\s+|then\s+|subsequently\s+|finally\s+)?(?:"
    + OPERATION_WORD_RE +
    r")\b)",
    flags=re.I,
)


def split_with_offsets(text, split_re):
    """Split text on split_re, keeping the character offsets of each piece."""
    if not isinstance(text, str) or not text.strip():
        return []

    pieces = []
    last = 0

    for match in list(split_re.finditer(text)) + [None]:
        start = last
        end = match.start() if match is not None else len(text)
        raw = text[start:end]
        stripped = raw.strip()

        if stripped:
            pieces.append({
                "text": stripped,
                "start": start + len(raw) - len(raw.lstrip()),
                "end": end - (len(raw) - len(raw.rstrip())),
            })

        if match is None:
            break

        last = match.end()

    return pieces



# Vessel vocabulary, generated from the training annotations exactly like the operation
# terms: every annotated vessel_text_span (rule R-T2), plus the head noun of each, so an
# unseen qualifier ("Teflon autoclave") is still recognised. Cell 10 reads it. Empty, and
# the vessel rule inert, when positive_operation_events.csv predates the vessel column.


def load_training_vessel_terms(csv_path=OPERATION_TERMS_CSV):
    operation_df = pd.read_csv(Path(csv_path), dtype=str).fillna("")

    if "vessel_text_span" not in operation_df.columns:
        return []

    terms = {
        re.sub(r"\s+", " ", value).strip(" .,;:?").lower()
        for value in operation_df["vessel_text_span"]
    }
    terms.discard("")

    return sorted(terms, key=lambda term: (-len(term), term))


TRAINED_VESSEL_TERMS = load_training_vessel_terms()
VESSEL_HEAD_NOUNS = sorted({term.split()[-1] for term in TRAINED_VESSEL_TERMS})

# Whole training phrases first, so "teflon-lined autoclave" wins over its head noun.
VESSEL_RE = (
    re.compile(
        r"\b(?:"
        + "|".join(operation_term_pattern(term) for term in TRAINED_VESSEL_TERMS)
        + "|"
        + "|".join(map(re.escape, VESSEL_HEAD_NOUNS))
        + r")\b",
        flags=re.I,
    )
    if TRAINED_VESSEL_TERMS
    else None
)

## Cell 5 — Heating-rate detection and sentence-level value detection

In [ ]:
# Cell 5: tell a set point from a ramp rate, and answer whether a sentence carries
# a usable temperature or time at all.
#
# The rate test is used twice: here, to gate sentences, and in Cell 13, to drop
# trained-model rows whose extracted temperature is really a ramp. It reads only
# the characters immediately before and after the value, never a window around it:
# a set point is routinely written next to its own ramp, as in "calcined at 600 °C
# (5 °C/min) for 4 h", so a rate merely being nearby says nothing about this value.

RATE_RIGHT_RE = re.compile(
    r"^\s*(?:/|per|·?\s*)\s*(?:min|mins|minute|minutes|h|hr|hour|hours)\s*(?:-?1|−1|⁻1)?\b",
    flags=re.I,
)

RATE_LEFT_RE = re.compile(
    r"(?:heating|cooling|ramp(?:ing)?|scan|flow|increase|temperature)\s+rate\s*(?:of\s*)?$",
    flags=re.I,
)


def is_temperature_rate(text, start, end):
    """True if the temperature at [start, end) is a ramp rate, not a set point."""
    left = text[max(0, start - 80):start]
    right = text[end:min(len(text), end + 50)]

    return bool(RATE_LEFT_RE.search(left) or RATE_RIGHT_RE.search(right))


def sentence_has_temperature(sentence):
    """True if the sentence states a temperature that is neither a range nor a rate.

    The bare-degree pass is the reason clause splitting is still needed: "60°" is
    only trusted when it sits in a clause that names an operation.
    """
    text = normalize_text_for_operation_snowball(sentence)

    if not text:
        return False

    used_spans = []

    for match in TEMP_RE.finditer(text):
        used_spans.append((match.start(), match.end()))

        if TEMP_RANGE_VALUE_RE.match(str(match.group("value"))):
            continue

        if not is_temperature_rate(text, match.start(), match.end()):
            return True

    for clause in split_with_offsets(text, CLAUSE_SPLIT_RE):
        if OPERATION_CUE_RE.search(clause["text"]) is None:
            continue

        for match in BARE_DEGREE_RE.finditer(clause["text"]):
            start = clause["start"] + match.start()
            end = clause["start"] + match.end()

            if any(start < used_end and end > used_start for used_start, used_end in used_spans):
                continue

            if not is_temperature_rate(text, start, end):
                return True

    return False


def sentence_has_time(sentence):
    text = normalize_text_for_operation_snowball(sentence)
    return bool(text) and TIME_RE.search(text) is not None

## Cell 6 — Operation-sentence selection

In [ ]:
# Cell 6: choose the sentences the parser sees.
#
# A sentence is kept only if it names an operation and states a temperature or a
# time, and is dropped if it describes characterization or catalytic testing. The
# analytical keywords below were added after error analysis: they remove 18 false
# positive records and zero true positives on this corpus. They matter because
# such protocols identify themselves either by an abbreviation (TPD, TPR) or by
# the material examined ("spent samples") rather than by the technique name.

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

CHARACTERIZATION_SENTENCE_RE = re.compile(
    r"\b(?:"
    r"characteri[sz]ation|characteri[sz]ed|"
    r"xrd|x-ray diffraction|x ray diffraction|"
    r"sem|tem|hrtem|stem|eds|edx|"
    r"xps|raman|ftir|ft-ir|uv-vis|uv vis|"
    r"bet|brunauer|n2 adsorption|n 2 adsorption|nitrogen adsorption|"
    r"tga|dtg|dsc|tg-dsc|"
    r"h2-tpr|h 2-tpr|temperature-programmed reduction|"
    r"co2-tpd|co 2-tpd|nh3-tpd|nh 3-tpd|"
    r"icp|icp-oes|icp-ms|"
    r"catalytic activity|activity test|reaction test|performance test|"
    r"catalyst testing|evaluation|measurement|"
    r"tp[rdo]|"
    r"diffractogram(?:s)?|"
    r"chemisorption|"
    r"thermogravimetric|"
    r"(?:spent|used)\s+(?:catalyst|catalysts|sample|samples)"
    r")\b"
    r"|\bcoke\b[^.]{0,60}\b(?:quantif|deposit)",
    flags=re.I,
)


def classify_sentence_for_operation_input(sentence):
    """Return (keep, status, has_operation_cue, has_temperature, has_time)."""
    if not isinstance(sentence, str) or not sentence.strip():
        return False, "empty_sentence", False, False, False

    sentence = normalize_text_for_operation_snowball(sentence)

    has_operation_cue = OPERATION_CUE_RE.search(sentence) is not None
    has_temperature = sentence_has_temperature(sentence)
    has_time = sentence_has_time(sentence)

    if CHARACTERIZATION_SENTENCE_RE.search(sentence) is not None:
        status = "removed_characterization_or_testing_sentence"
    elif not has_operation_cue:
        status = "removed_no_operation_cue"
    elif not has_temperature and not has_time:
        status = "removed_no_temperature_or_time"
    else:
        return True, "kept_operation_sentence", has_operation_cue, has_temperature, has_time

    return False, status, has_operation_cue, has_temperature, has_time


def build_operation_sentence_input(filtered_text):
    """Split a filtered synthesis section into the sentences the parser will read."""
    if not isinstance(filtered_text, str) or not filtered_text.strip():
        return [], []

    filtered_text = normalize_text_for_operation_snowball(filtered_text)

    kept_sentences = []
    audit_rows = []

    for sentence_index, item in enumerate(split_with_offsets(filtered_text, SENTENCE_SPLIT_RE)):
        sentence = item["text"]
        keep, status, has_operation, has_temperature, has_time = (
            classify_sentence_for_operation_input(sentence)
        )

        audit_rows.append({
            "operation_sentence_index": sentence_index,
            "operation_sentence_status": status,
            "operation_sentence_kept": keep,
            "operation_sentence_text": sentence,
            "operation_sentence_start": item["start"],
            "operation_sentence_end": item["end"],
            "operation_sentence_has_operation_cue": has_operation,
            "operation_sentence_has_temperature": has_temperature,
            "operation_sentence_has_time": has_time,
        })

        if keep:
            kept_sentences.append(sentence)

    return kept_sentences, audit_rows

## Cell 7 — Attach the trained Snowball2 parsers

In [ ]:
# Cell 7: load the phrase lists written by notebook 001 and attach them to the models.
#
# Any Snowball parser already on the model class is replaced, so re-running this
# cell cannot stack two runtime parsers on one model.


def attach_snowball_parser_to_model(
    model_cls,
    checkpoint_path,
    runtime_file_name,
    confidence_limit=CONFIDENCE_LIMIT,
    tsim_h=0.95,
    tsim_l=0.5,
    tc=0.9,
):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Snowball checkpoint not found for {model_cls.__name__}: {checkpoint_path}"
        )

    kwargs = {
        "model": model_cls,
        "tsim_h": tsim_h,
        "tsim_l": tsim_l,
        "tc": tc,
        "file_name": runtime_file_name,
        "save_dir": str(RUNTIME_SAVE_DIR),
        "clustering_method": "max",
    }

    try:
        sb_operation = Snowball(**kwargs)
    except TypeError:
        kwargs.pop("clustering_method", None)
        sb_operation = Snowball(**kwargs)

    sb_operation.import_from_list(path=str(checkpoint_path), confidence_limit=confidence_limit)

    try:
        sb_operation.cluster_all(method="max")
    except TypeError:
        sb_operation.cluster_all()

    model_cls.parsers = [
        parser for parser in getattr(model_cls, "parsers", [])
        if not isinstance(parser, Snowball)
    ] + [sb_operation]

    return sb_operation


def attach_operation_snowball_parsers():
    attached = {
        OperationTemperature.__name__: attach_snowball_parser_to_model(
            model_cls=OperationTemperature,
            checkpoint_path=OPERATION_TEMPERATURE_CHECKPOINT,
            runtime_file_name=RUNTIME_TEMPERATURE_FILE_NAME,
        )
    }

    if ENABLE_TIME_SNOWBALL_MODEL:
        attached[OperationTime.__name__] = attach_snowball_parser_to_model(
            model_cls=OperationTime,
            checkpoint_path=OPERATION_TIME_CHECKPOINT,
            runtime_file_name=RUNTIME_TIME_FILE_NAME,
        )

    return attached

## Cell 8 — Sentence parsing

In [ ]:
# Cell 8: hand one sentence to ChemDataExtractor and get its records back.
#
# The tokenization repair must stay identical to notebook 001 Part 3 Cell 6. The
# vendored tokenizer leaves a number glued to its unit ("4h", "30min") as ONE
# token, so "4h" can never match a pattern learned from the tokens "4" + "h".
# Notebook 001 splits them before building a training phrase; without the same
# split here those patterns never fire at inference. Single-letter units are
# matched case-sensitively so "1H NMR", "2D" and "H2O2" are left alone.

GLUED_VALUE_UNIT_RE = re.compile(
    r"(?<=\d)"
    r"(?=(?:°\s*C|℃|º\s*C|oC|K|"
    r"hrs|hr|hours|hour|h|"
    r"mins|min|minutes|minute|"
    r"secs|sec|seconds|second|s|"
    r"days|day|d)"
    r"(?![A-Za-z0-9]))"
)


def normalize_for_tokenization(text):
    """Insert the space the tokenizer will not insert itself (see above)."""
    return GLUED_VALUE_UNIT_RE.sub(" ", "" if text is None else str(text))


def extract_records_from_sentence(sentence, models=None):
    models = OPERATION_MODELS if models is None else models
    document = Document(Paragraph(normalize_for_tokenization(sentence.strip())), models=models)
    document.models = models

    return document.records

## Cell 9 — Record flattening

In [ ]:
# Cell 9: turn a serialized record into one flat dictionary of columns.
#
# A record nests its compound under compound.Compound.names, so nested keys are
# joined with a dot and any list is stored as JSON text.


def get_cell_text(row, column_name):
    value = row.get(column_name, "")
    return "" if pd.isna(value) else str(value).strip()


def flatten_dict(d, parent_key="", sep="."):
    if not isinstance(d, dict):
        return {"value": d}

    flattened = {}

    for key, value in d.items():
        new_key = f"{parent_key}{sep}{key}" if parent_key else str(key)

        if isinstance(value, dict):
            flattened.update(flatten_dict(value, new_key, sep=sep))
        elif isinstance(value, list):
            flattened[new_key] = json.dumps(value, ensure_ascii=False)
        else:
            flattened[new_key] = value

    return flattened


def serialized_records_to_rows(serialized):
    """records.serialize() returns a list of one-key dicts, one key per model."""
    if serialized is None:
        return []

    items = serialized if isinstance(serialized, list) else [serialized]

    return [
        {"record_index": index, **flatten_dict(item)}
        for index, item in enumerate(items)
    ]


def make_base_metadata(article_id, row_index, source_year, section_filter_removed_heading=None):
    return {
        "article_id": article_id,
        "source_row_index": int(row_index),
        "source_excel_row": int(row_index) + 2,
        "source_year": source_year,
        "text_column": TEXT_COLUMN,
        "extraction_target": "operation_temperature_time",
        "section_filter_removed_heading": section_filter_removed_heading,
    }

## Cell 10 — Parser-ready columns

In [ ]:
# Cell 10: map the parser's own field names onto the pipeline's column names.
#
# OperationTemperature and OperationTime are declared in operation_model.py as
# TemperatureModel / TimeModel subclasses, so a record carries exactly raw_value,
# raw_units, value, units, specifier and an optional compound. After flattening,
# those sit under one "OperationTemperature." or "OperationTime." prefix, and the
# prefix alone says which model produced the record. A record never carries both.

TEMPERATURE_PREFIX = "OperationTemperature."
TIME_PREFIX = "OperationTime."


def model_prefix_of(row):
    has_temperature = any(str(key).startswith(TEMPERATURE_PREFIX) for key in row)
    has_time = any(str(key).startswith(TIME_PREFIX) for key in row)

    if has_temperature and not has_time:
        return TEMPERATURE_PREFIX

    if has_time and not has_temperature:
        return TIME_PREFIX

    return None


def clean_field(value):
    """Empty strings, NaN and the literal 'nan' all mean 'not extracted'."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None

    text = str(value).strip()

    return None if not text or text.lower() == "nan" else value


def time_surface_from_row(row):
    value = row.get("time_raw_value")
    unit = row.get("time_raw_units")

    return None if value is None or unit is None else f"{value} {unit}".strip()


def find_event_vessel(sentence, specifier):
    """(vessel text, sealed) for the step this trigger names, or (None, False).

    Only the trigger's own clause is read, because one sentence often carries the step
    before the vessel step and the one after it. The clause splitter cuts before "and
    kept", so in "transferred into an autoclave and kept at 180 °C" the vessel sits in the
    clause before; that clause is read too, but only when it names no operation itself.
    """
    if VESSEL_RE is None or not isinstance(sentence, str) or specifier is None:
        return None, False

    text = normalize_text_for_operation_snowball(sentence)
    pattern = operation_term_pattern(specifier)
    trigger = re.search(r"\b(?:" + pattern + r")\b", text, flags=re.I) if pattern else None

    if trigger is None:
        return None, False

    clauses = split_with_offsets(text, CLAUSE_SPLIT_RE)

    for index, clause in enumerate(clauses):
        if not clause["start"] <= trigger.start() < clause["end"]:
            continue

        candidates = [clause]

        if index > 0 and OPERATION_CUE_RE.search(clauses[index - 1]["text"]) is None:
            candidates.append(clauses[index - 1])

        for candidate in candidates:
            match = VESSEL_RE.search(candidate["text"])

            if match:
                sealed = bool(
                    SEALED_VESSEL_RE.search(match.group(0))
                    or re.search(r"\bsealed\b", candidate["text"], flags=re.I)
                )
                return match.group(0), sealed

        break

    return None, False


def apply_sealed_vessel_rule(row):
    """Record the step's vessel, and apply rule R-T3 to its operation label."""
    vessel, sealed = find_event_vessel(row.get("record_sentence"), row.get("operation_specifier"))
    row["vessel_text"] = vessel

    if (
        ENABLE_SEALED_VESSEL_RULE
        and sealed
        and row.get("operation_specifier_final") in SEALED_VESSEL_RULE_OPERATIONS
    ):
        row["operation_specifier_final"] = SEALED_VESSEL_OPERATION
        row["operation_specifier_final_source"] = "sealed_vessel_rule"

    return row


def add_parser_ready_columns(row):
    prefix = model_prefix_of(row)
    kind = {TEMPERATURE_PREFIX: "temperature", TIME_PREFIX: "time"}.get(prefix)

    def field(name):
        return None if prefix is None else clean_field(row.get(prefix + name))

    row["parser_model_kind"] = kind
    row["operation_specifier"] = field("specifier")
    row["operation_specifier_normalized"] = normalize_specifier(row["operation_specifier"])
    row["operation_specifier_final"] = row["operation_specifier_normalized"]
    row["operation_specifier_final_source"] = (
        "normalized_model_specifier"
        if row["operation_specifier_normalized"] is not None
        else None
    )

    is_temperature = kind == "temperature"
    is_time = kind == "time"

    row["temperature_raw_value"] = field("raw_value") if is_temperature else None
    row["temperature_raw_units"] = field("raw_units") if is_temperature else None
    row["temperature_normalized_value"] = field("value") if is_temperature else None
    row["temperature_normalized_units"] = field("units") if is_temperature else None

    row["time_raw_value"] = field("raw_value") if is_time else None
    row["time_raw_units"] = field("raw_units") if is_time else None

    row["material_or_compound"] = field("compound.Compound.names")
    row["record_sentence"] = field("sentence") or row.get("source_text")
    row["record_confidence"] = field("confidence")

    apply_sealed_vessel_rule(row)

    has_time_pair = row["time_raw_value"] is not None and row["time_raw_units"] is not None
    row["time_surface_model"] = time_surface_from_row(row) if has_time_pair else None
    row["time_source"] = "cde_snowball_time_model" if has_time_pair else None

    return row

## Cell 11 — Corpus loading

In [ ]:
# Cell 11: build the paper-level corpus from the validation workbook.
#
# Papers are identified by ID, which the validation sheet carries on every row; title
# and paragraph_text are written only on a paper's first row, so a paper's text
# is the concatenation of its block.


def normalize_year_value(value):
    if value is None or pd.isna(value):
        return None

    match = re.search(r"(19|20)\d{2}", str(value))

    return match.group(0) if match else None


def load_corpus_from_validation(validation_file=None, validation_sheet=None):
    validation_file = VALIDATION_FILE if validation_file is None else validation_file
    validation_sheet = VALIDATION_SHEET if validation_sheet is None else validation_sheet

    validation = pd.read_excel(validation_file, sheet_name=validation_sheet, dtype=str).fillna("")

    for required in ["ID", "title", "year", "paragraph_text"]:
        if required not in validation.columns:
            raise ValueError(f"Validation sheet is missing the '{required}' column")

    papers = []

    for paper_id, block in validation.groupby("ID", sort=False):
        paragraphs = [str(v).strip() for v in block["paragraph_text"] if str(v).strip()]
        titles = [str(v).strip() for v in block["title"] if str(v).strip()]
        years = [str(v).strip() for v in block["year"] if str(v).strip()]

        papers.append({
            "ID": str(paper_id).strip(),
            "Title": titles[0] if titles else "",
            "year": years[0] if years else "",
            TEXT_COLUMN: "\n\n".join(paragraphs),
        })

    corpus = pd.DataFrame(papers)

    empty = corpus.loc[corpus[TEXT_COLUMN].str.strip() == "", "ID"]

    if len(empty):
        print("Papers with no paragraph_text in the validation file:", ", ".join(empty))

    print(f"Corpus loaded from {validation_file} [{validation_sheet}]: {len(corpus)} papers")

    return corpus

## Cell 12 — Bind time records to temperature records

In [ ]:
# Cell 12: decide which OperationTime record belongs to which OperationTemperature
# record, within one sentence.
#
# Pairs are scored and then consumed best-first, so one duration can never be
# reported on two temperatures. A time record that no temperature claims is
# returned separately and becomes a time-only event.


def normalized_binding_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    return re.sub(r"\s+", " ", normalize_text_for_operation_snowball(text)).strip().lower()


def find_first_surface_position(text, surfaces):
    text_norm = normalized_binding_text(text)

    if not text_norm:
        return None

    for surface in surfaces:
        surface_norm = normalized_binding_text(str(surface)) if surface is not None else ""

        if not surface_norm:
            continue

        position = text_norm.find(surface_norm)

        if position >= 0:
            return position

    return None


def row_sentence_for_time_binding(row):
    return row.get("record_sentence") or row.get("source_text") or ""


def sentences_overlap_for_time_binding(sentence_a, sentence_b):
    a = normalized_binding_text(sentence_a)
    b = normalized_binding_text(sentence_b)

    return bool(a) and bool(b) and (a == b or a in b or b in a)


def score_time_model_row_for_temperature_row(temperature_row, time_row):
    """Lower is better. None means the two records are not in the same sentence."""
    temperature_sentence = row_sentence_for_time_binding(temperature_row)

    if not sentences_overlap_for_time_binding(
        temperature_sentence,
        row_sentence_for_time_binding(time_row),
    ):
        return None

    temp_position = find_first_surface_position(
        temperature_sentence,
        [temperature_row.get("temperature_surface"), temperature_row.get("temperature_raw_value")],
    )

    time_surfaces = [
        time_row.get("time_surface_model"),
        time_row.get("time_raw_value"),
    ]

    if time_row.get("time_raw_value") is not None and time_row.get("time_raw_units") is not None:
        time_surfaces.insert(0, f"{time_row.get('time_raw_value')} {time_row.get('time_raw_units')}")

    time_position = find_first_surface_position(temperature_sentence, time_surfaces)

    if temp_position is None or time_position is None:
        distance_score = 500
    else:
        distance_score = abs(time_position - temp_position)

        # Slightly prefer "500 °C for 2 h" over a time written before the temperature.
        if time_position >= temp_position:
            distance_score -= 10

    temperature_event = normalize_specifier(temperature_row.get("operation_specifier_final"))
    time_event = normalize_specifier(time_row.get("operation_specifier_final"))

    if temperature_event and time_event:
        distance_score += -50 if temperature_event == time_event else 150

    return distance_score


def bind_time_rows_to_temperature_rows(temperature_rows, time_rows):
    """Return (pairs, unbound_time_rows) with a one-to-one assignment.

    pairs holds (temperature_row, bound_time_row_or_None). Only a row left in
    unbound_time_rows may be emitted as a standalone time-only event.
    """
    usable_time_rows = [
        time_row for time_row in time_rows
        if time_row.get("time_raw_value") is not None
        and time_row.get("time_raw_units") is not None
    ]

    if not usable_time_rows:
        return [(row.copy(), None) for row in temperature_rows], []

    scored_pairs = []

    for temperature_index, temperature_row in enumerate(temperature_rows):
        for time_index, time_row in enumerate(usable_time_rows):
            score = score_time_model_row_for_temperature_row(temperature_row, time_row)

            if score is not None:
                scored_pairs.append((score, temperature_index, time_index))

    scored_pairs.sort()

    assignment = {}
    used_time_indices = set()

    for score, temperature_index, time_index in scored_pairs:
        if temperature_index in assignment or time_index in used_time_indices:
            continue

        assignment[temperature_index] = (score, time_index)
        used_time_indices.add(time_index)

    pairs = []

    for temperature_index, temperature_row in enumerate(temperature_rows):
        if temperature_index not in assignment:
            pairs.append((temperature_row.copy(), None))
            continue

        score, time_index = assignment[temperature_index]
        time_row = usable_time_rows[time_index]

        output_row = temperature_row.copy()
        output_row["time_raw_value"] = time_row.get("time_raw_value")
        output_row["time_raw_units"] = time_row.get("time_raw_units")
        output_row["time_surface_model"] = time_surface_from_row(time_row)
        output_row["time_source"] = "cde_snowball_time_model"
        output_row["time_binding_rule"] = "one_to_one_operationtime_model_record_same_sentence"
        output_row["time_binding_confidence"] = 0.95 if score <= 150 else 0.80
        output_row["time_model_record_index"] = time_row.get("record_index")
        output_row["time_model_binding_score"] = score
        output_row["time_model_sentence"] = time_row.get("record_sentence")

        pairs.append((output_row, time_row))

    unbound_time_rows = [
        time_row for time_index, time_row in enumerate(usable_time_rows)
        if time_index not in used_time_indices
    ]

    return pairs, unbound_time_rows

## Cell 13 — Heating-rate false-positive filter

In [ ]:
# Cell 13: drop a trained-model row whose extracted temperature is a ramp rate.
#
# The model reports a value and a unit but not an offset, so the value is located
# back in its own sentence and the Cell 5 rate test is applied there. Examples
# this removes: 1 °C/min, 3 °C min-1, 7.5 °C/min.


def temperature_unit_to_regex(unit):
    unit = normalize_temperature_unit(unit)

    if unit == "°C":
        return r"(?:°\s*C|℃|deg\s*C|degrees?\s*C|celsius)"

    if unit == "°F":
        return r"(?:°\s*F|℉|fahrenheit)"

    if unit == "K":
        return r"(?:K|kelvin)"

    return re.escape(str(unit))


def model_row_temperature_is_rate(row):
    sentence = row.get("record_sentence") or row.get("source_text") or ""

    if not isinstance(sentence, str) or not sentence.strip():
        return False

    text = normalize_text_for_operation_snowball(sentence)

    temp_value = canonical_temperature_value(row.get("temperature_raw_value"))
    temp_unit = normalize_temperature_unit(row.get("temperature_raw_units"))

    if temp_value is None or temp_unit is None:
        return False

    unit_pattern = temperature_unit_to_regex(temp_unit)

    for value in [part.strip() for part in str(temp_value).split(" to ") if part.strip()]:
        pattern = re.compile(
            r"(?<![\d.])" + re.escape(value) + r"\s*" + unit_pattern + r"\b",
            flags=re.I,
        )

        for match in pattern.finditer(text):
            if is_temperature_rate(text, match.start(), match.end()):
                return True

    return False



# Heating rates. A rate is never an event of its own: it is a parameter of the set point
# it ramps to ("calcined at 600 °C (5 °C/min) for 4 h", "heated at a rate of 2 °C/min to
# 500 °C"). The rates of a sentence are the temperatures the Cell 5 test calls rates and
# that carry a time unit. A rate belongs to the set point written nearest to it, so it is
# bound only to an event with that temperature, one to one, the way Cell 12 binds
# durations; a rate whose set point is no event ("sintered at 1000 °C (3 °C/min)" with
# no sintering event) stays unbound rather than falling to another step. Rows that differ
# only in the compound are one event, so they share the rate and still collapse in Cell 16.

RATE_TIME_UNIT_RE = re.compile(
    r"^\s*(?:/|per|·)?\s*(?P<per>min|mins|minute|minutes|h|hr|hrs|hour|hours)\b",
    flags=re.I,
)


def find_heating_rates(sentence):
    """(position, value, units, anchor) for each heating rate in the sentence.

    units reads like '°C/min'. anchor is the (value, unit) of the set point written
    nearest to the rate: the only temperature the rate may be attached to.
    """
    text = normalize_text_for_operation_snowball(sentence) if isinstance(sentence, str) else ""
    rates, set_points = [], []

    for match in TEMP_RE.finditer(text or ""):
        if TEMP_RANGE_VALUE_RE.match(str(match.group("value"))):
            continue

        value = canonical_temperature_value(match.group("value"))
        unit = normalize_temperature_unit(match.group("unit"))

        if not is_temperature_rate(text, match.start(), match.end()):
            set_points.append((match.start(), (value, unit)))
            continue

        per = RATE_TIME_UNIT_RE.match(text[match.end():])

        if per is None:
            continue

        per_unit = "min" if per.group("per").lower().startswith("min") else "h"
        rates.append((match.start(), value, f"{unit}/{per_unit}"))

    return [
        (position, value, units,
         min(set_points, key=lambda point: abs(point[0] - position))[1] if set_points else None)
        for position, value, units in rates
    ]


def set_point_position(text, row):
    """Where the row's temperature stands in the text, skipping any occurrence that is a rate."""
    value = canonical_temperature_value(row.get("temperature_raw_value"))
    unit = normalize_temperature_unit(row.get("temperature_raw_units"))

    if value is None or unit is None:
        return None

    pattern = re.compile(
        r"(?<![\d.])" + re.escape(str(value).split(" to ")[0]) + r"\s*"
        + temperature_unit_to_regex(unit) + r"\b",
        flags=re.I,
    )

    for match in pattern.finditer(text):
        if not is_temperature_rate(text, match.start(), match.end()):
            return match.start()

    return None


def bind_heating_rates(event_rows):
    """Write heating_rate_value / heating_rate_units onto the events, in place.

    Returns the number of rates bound."""
    events = {}

    for row in event_rows:
        row["heating_rate_value"] = None
        row["heating_rate_units"] = None

        if row.get("temperature_raw_value") is None:
            continue

        sentence = row.get("record_sentence") or row.get("source_text") or ""
        sentence_key = (row.get("operation_sentence_index"), sentence)
        event_key = tuple(
            str(row.get(column)) for column in (
                "operation_specifier_final", "temperature_raw_value", "temperature_raw_units",
                "time_raw_value", "time_raw_units",
            )
        )
        events.setdefault(sentence_key, {}).setdefault(event_key, []).append(row)

    bound = 0

    for (_, sentence), sentence_events in events.items():
        rates = find_heating_rates(sentence)

        if not rates:
            continue

        text = normalize_text_for_operation_snowball(sentence)
        groups = list(sentence_events.values())
        scored = []

        for group_index, rows in enumerate(groups):
            position = set_point_position(text, rows[0])

            if position is None:
                continue

            set_point = (
                canonical_temperature_value(rows[0].get("temperature_raw_value")),
                normalize_temperature_unit(rows[0].get("temperature_raw_units")),
            )

            for rate_index, (rate_position, _, _, anchor) in enumerate(rates):
                if anchor == set_point:
                    scored.append((abs(rate_position - position), group_index, rate_index))

        scored.sort()
        used_groups, used_rates = set(), set()

        for _, group_index, rate_index in scored:
            if group_index in used_groups or rate_index in used_rates:
                continue

            _, value, units, _ = rates[rate_index]

            for row in groups[group_index]:
                row["heating_rate_value"] = value
                row["heating_rate_units"] = units

            used_groups.add(group_index)
            used_rates.add(rate_index)
            bound += 1

    return bound


# Rate rows. The model sometimes reports a ramp rate as the step's temperature. Such a row
# is settled before any duration is bound (Cell 14), so a rate row can never take the
# duration of the step. A row whose trigger is only the "heating" of "heating rate" names
# no step and is dropped. Any other rate row is the step itself read at the wrong number:
# its temperature becomes the set point the rate belongs to (the anchor of Cell 13's rate
# test), unless the parser already reported that set point in the same sentence.

RATE_PHRASE_TAIL_RE = re.compile(r"[\s-]+(?:rate|ramp)\b", flags=re.I)


def specifier_is_rate_phrase(row):
    """True when every occurrence of the trigger in its sentence reads 'heating rate'."""
    specifier = row.get("operation_specifier")
    sentence = row.get("record_sentence") or row.get("source_text") or ""

    if not specifier or not isinstance(sentence, str):
        return False

    text = normalize_text_for_operation_snowball(sentence)
    occurrences = list(
        re.finditer(r"\b" + re.escape(str(specifier).strip()) + r"\b", text, flags=re.I)
    )

    return bool(occurrences) and all(
        RATE_PHRASE_TAIL_RE.match(text[match.end():]) for match in occurrences
    )


def rate_set_point(row):
    """(value, unit) of the set point the row's rate ramps to, or None."""
    sentence = row.get("record_sentence") or row.get("source_text") or ""
    value = canonical_temperature_value(row.get("temperature_raw_value"))

    for _, rate_value, _, anchor in find_heating_rates(sentence):
        if rate_value == value:
            return anchor

    return None


def settle_model_temperature_rates(temperature_rows):
    """Return (kept rows, dropped rows); kept rows carry a set point, never a rate."""
    kept, rate_rows = [], []

    for row in temperature_rows:
        if model_row_temperature_is_rate(row):
            rate_rows.append(row)
        else:
            row["temperature_rate_filter_status"] = "kept"
            kept.append(row)

    reported = {
        (
            row.get("operation_sentence_index"),
            canonical_temperature_value(row.get("temperature_raw_value")),
            normalize_temperature_unit(row.get("temperature_raw_units")),
        )
        for row in kept
    }
    dropped = []

    for row in rate_rows:
        anchor = None if specifier_is_rate_phrase(row) else rate_set_point(row)
        key = None if anchor is None else (row.get("operation_sentence_index"), *anchor)

        if key is None or key in reported:
            row["temperature_rate_filter_status"] = "dropped_model_temperature_rate"
            dropped.append(row)
            continue

        row["temperature_raw_value"], row["temperature_raw_units"] = anchor
        row["temperature_normalized_value"] = None
        row["temperature_normalized_units"] = None
        row["temperature_rate_filter_status"] = "set_point_of_rate"
        reported.add(key)
        kept.append(row)

    return kept, dropped

## Cell 14 — Extraction run

In [ ]:
# Cell 14: one pass over the corpus, producing the long record table and the audits.
#
# Each kept sentence is parsed on its own. Parsing a whole block in one call left
# every record carrying the same article-wide text, which made the sentence gate
# in Cell 12 a no-op and measured binding distance across the whole article.

EXTRACTION_METHOD = "cde_snowball_model_sentence_input"


def run_extraction():
    save_root = Path(SAVE_ROOT_DIR)
    save_root.mkdir(parents=True, exist_ok=True)

    attached_parsers = attach_operation_snowball_parsers()
    corpus_df = load_corpus_from_validation()

    combined_rows = []
    combined_by_article = {}
    operation_sentence_audit_rows = []
    model_time_audit_rows = []
    dropped_model_rate_rows = []
    heating_rates_bound = 0

    for row_index, row in corpus_df.iterrows():
        original_text = get_cell_text(row, TEXT_COLUMN)

        if not original_text:
            continue

        source_year = normalize_year_value(row.get("year"))
        filtered_text = original_text
        removed_heading = None

        # The section cut runs on the raw text: it finds headings by their line
        # breaks, which normalization collapses into spaces.
        if ENABLE_SECTION_FILTER:
            filtered_text, removed_heading = filter_text_after_stop_heading(filtered_text)

        filtered_text = normalize_text_for_operation_snowball(filtered_text)

        year_prefix = f"{source_year}_" if source_year else ""
        article_id = f"{year_prefix}row_{row_index + 1}_operation_temperature_time_{TEXT_COLUMN}"

        base_metadata = make_base_metadata(
            article_id=article_id,
            row_index=row_index,
            source_year=source_year,
            section_filter_removed_heading=removed_heading,
        )

        kept_sentences, sentence_audit_rows = build_operation_sentence_input(filtered_text)

        for sentence_audit_row in sentence_audit_rows:
            sentence_audit_row.update(base_metadata)
            operation_sentence_audit_rows.append(sentence_audit_row)

        serialized = []
        parser_rows_for_article = []

        for sentence_index, sentence_text in enumerate(kept_sentences):
            sentence_records = extract_records_from_sentence(sentence_text)
            sentence_serialized = sentence_records.serialize()

            if isinstance(sentence_serialized, list):
                serialized.extend(sentence_serialized)
            else:
                serialized.append(sentence_serialized)

            for extracted_row in serialized_records_to_rows(sentence_serialized):
                parser_rows_for_article.append(add_parser_ready_columns({
                    **base_metadata,
                    # record_sentence falls back to source_text, so this is what
                    # makes the Cell 12 sentence gate discriminate between events.
                    "source_text": sentence_text,
                    "operation_sentence_index": sentence_index,
                    "source_text_original": original_text,
                    "source_text_filtered": filtered_text,
                    "extraction_method": EXTRACTION_METHOD,
                    **extracted_row,
                }))

            if RUN_VERBOSE:
                print("Article:", article_id, "| sentence", sentence_index, sentence_serialized)

        combined_by_article[article_id] = {
            **base_metadata,
            "models": [model.__name__ for model in OPERATION_MODELS],
            "attached_snowball_parsers": list(attached_parsers.keys()),
            "source_text_original": original_text,
            "source_text_filtered": filtered_text,
            "operation_sentences": kept_sentences,
            "records": serialized,
        }

        model_time_rows = [
            parser_row for parser_row in parser_rows_for_article
            if parser_row.get("parser_model_kind") == "time"
            and parser_row.get("time_raw_value") is not None
            and parser_row.get("time_raw_units") is not None
        ]

        for model_time_row in model_time_rows:
            audit_row = model_time_row.copy()
            audit_row["time_model_audit_status"] = "available_for_binding"
            model_time_audit_rows.append(audit_row)

        model_temperature_rows = [
            parser_row for parser_row in parser_rows_for_article
            if parser_row.get("parser_model_kind") == "temperature"
            and parser_row.get("temperature_raw_value") is not None
            and parser_row.get("temperature_raw_units") is not None
        ]

        # Rates are settled before durations are bound (Cell 13), so a duration is
        # only ever paired with a set point.
        if DROP_MODEL_TEMPERATURE_RATES:
            model_temperature_rows, rate_rows = settle_model_temperature_rates(
                model_temperature_rows
            )
            dropped_model_rate_rows.extend(rate_rows)

        bound_pairs, unbound_time_rows = bind_time_rows_to_temperature_rows(
            temperature_rows=model_temperature_rows,
            time_rows=model_time_rows,
        )

        article_event_rows = []

        for output_row, bound_time_row in bound_pairs:
            output_row.setdefault("temperature_rate_filter_status", "kept")
            output_row["record_status"] = "temperature_event"
            article_event_rows.append(output_row)

        # Only a time record that no temperature claimed becomes a standalone row.
        if KEEP_MODEL_TIME_ONLY_ROWS:
            for time_row in unbound_time_rows:
                time_row = time_row.copy()
                time_row["record_status"] = "time_only_event"
                article_event_rows.append(time_row)

        if ATTACH_HEATING_RATES:
            heating_rates_bound += bind_heating_rates(article_event_rows)

        combined_rows.extend(article_event_rows)

    combined_df = pd.DataFrame(combined_rows)

    combined_json_path = save_root / "combined_results_nested.json"
    combined_excel_path = save_root / "combined_results_long.xlsx"
    combined_csv_path = save_root / "combined_results_long.csv"

    with open(combined_json_path, "w", encoding="utf-8") as f:
        json.dump(combined_by_article, f, ensure_ascii=False, indent=2)

    combined_df.to_excel(combined_excel_path, index=False)
    combined_df.to_csv(combined_csv_path, index=False, encoding="utf-8-sig")

    audit_paths = {}

    for should_save, rows, name in [
        (SAVE_OPERATION_SENTENCE_AUDIT, operation_sentence_audit_rows, "operation_sentence_audit"),
        (SAVE_MODEL_TIME_AUDIT, model_time_audit_rows, "model_time_audit"),
        (bool(dropped_model_rate_rows), dropped_model_rate_rows, "dropped_model_temperature_rate_rows"),
    ]:
        if not should_save:
            audit_paths[name] = None
            continue

        path = save_root / f"{name}.xlsx"
        pd.DataFrame(rows).to_excel(path, index=False)
        audit_paths[name] = str(path)

    summary = {
        "validation_file": VALIDATION_FILE,
        "validation_sheet": VALIDATION_SHEET,
        "save_root_dir": str(save_root),
        "models": [model.__name__ for model in OPERATION_MODELS],
        "attached_snowball_parsers": list(attached_parsers.keys()),
        "temperature_checkpoint_path": str(OPERATION_TEMPERATURE_CHECKPOINT),
        "time_checkpoint_path": str(OPERATION_TIME_CHECKPOINT) if ENABLE_TIME_SNOWBALL_MODEL else None,
        "papers_in_corpus": int(len(corpus_df)),
        "articles_parsed": int(len(combined_by_article)),
        "extracted_record_rows": int(len(combined_rows)),
        "operation_sentence_audit_rows": int(len(operation_sentence_audit_rows)),
        "model_time_audit_rows": int(len(model_time_audit_rows)),
        "dropped_model_temperature_rate_rows": int(len(dropped_model_rate_rows)),
        "heating_rates_bound": heating_rates_bound,
        "sealed_vessel_rule_enabled": ENABLE_SEALED_VESSEL_RULE,
        "rows_relabelled_by_sealed_vessel_rule": sum(
            row.get("operation_specifier_final_source") == "sealed_vessel_rule"
            for row in combined_rows
        ),
        "section_filter_enabled": ENABLE_SECTION_FILTER,
        "time_snowball_model_enabled": ENABLE_TIME_SNOWBALL_MODEL,
        "time_assignment_policy": "OperationTime model only",
        "combined_nested_json": str(combined_json_path),
        "combined_long_excel": str(combined_excel_path),
        "combined_long_csv": str(combined_csv_path),
        **{f"{name}_excel": path for name, path in audit_paths.items()},
    }

    summary_path = save_root / "extraction_summary.json"

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("Extraction complete. Summary saved to:", summary_path)

    return summary, combined_df

## Cell 15 — Run

In [ ]:
summary, combined_df = run_extraction()
summary

## Cell 16 — Final event table

In [ ]:
# Cell 16: the one output notebook 003 reads.
#
# An event qualifies on a complete temperature OR a complete time, so a time-only
# event is reported with an empty temperature rather than dropped. Operations
# outside the trained set are tagged, not removed. The parser emits one record per
# compound in the sentence, which duplicates an otherwise identical event, so row
# identity ignores material and the materials seen for one event are joined.
# operation_sentence_index stays in the identity: a procedure sentence repeated
# verbatim for a second sample is a second event, not a duplicate record.

SAVE_ROOT = Path(SAVE_ROOT_DIR)
FINAL_OUTPUT_PATH = SAVE_ROOT / "operation_temperature_time_final_compact.xlsx"

FINAL_COLUMNS = [
    "article_title",
    "paper_id",
    "article_id",
    "source_excel_row",
    "source_year",
    "text_column",
    "operation_sentence_index",
    "operation_specifier_final",
    "temperature_raw_value",
    "temperature_raw_units",
    "time_raw_value",
    "time_raw_units",
    "heating_rate_value",
    "heating_rate_units",
    "time_source",
    "vessel_text",
    "material_or_compound",
    "extraction_method",
    "record_sentence",
]


def add_paper_identity_from_corpus(rows_df):
    """Title and ID are attached by position; the corpus and the validation set are one sheet."""
    corpus_df = load_corpus_from_validation()

    id_lookup = corpus_df["ID"].to_dict()
    title_lookup = corpus_df["Title"].to_dict()

    def lookup(table, index_value):
        return None if pd.isna(index_value) else table.get(int(index_value))

    rows_df = rows_df.copy()
    rows_df["paper_id"] = rows_df["source_row_index"].map(lambda i: lookup(id_lookup, i))
    rows_df["article_title"] = rows_df["source_row_index"].map(lambda i: lookup(title_lookup, i))

    return rows_df


final_df = add_paper_identity_from_corpus(combined_df)

has_temperature = final_df["temperature_raw_value"].notna() & final_df["temperature_raw_units"].notna()
has_time = final_df["time_raw_value"].notna() & final_df["time_raw_units"].notna()
named_operation = (
    final_df["operation_specifier_final"].notna()
    & (final_df["operation_specifier_final"].astype(str).str.lower() != "unknown")
)

final_df = final_df[named_operation & (has_temperature | has_time)].copy()
final_df = final_df[[col for col in FINAL_COLUMNS if col in final_df.columns]]

in_scope_mask = (
    final_df["operation_specifier_final"].astype(str).str.strip().str.lower().isin(IN_SCOPE_OPERATIONS)
)
final_df["operation_in_training_scope"] = in_scope_mask
out_of_scope_rows = final_df[~in_scope_mask]

if RESTRICT_TO_IN_SCOPE_OPERATIONS:
    final_df = final_df[in_scope_mask]

rows_before_material_collapse = len(final_df)

if COLLAPSE_MATERIAL_DUPLICATE_ROWS and "material_or_compound" in final_df.columns:
    def join_materials(values):
        seen = []

        for value in values:
            text = str(value).strip()

            if text and text.lower() not in {"nan", "none"} and text not in seen:
                seen.append(text)

        return "; ".join(seen) if seen else None

    def first_text(values):
        for value in values:
            text = str(value).strip()

            if text and text.lower() not in {"nan", "none"}:
                return text

        return None

    # The vessel describes the event, it does not identify it: two records of one
    # event, one of which found the vessel, are still one event.
    described_by = {"material_or_compound": join_materials, "vessel_text": first_text}
    described_by = {col: agg for col, agg in described_by.items() if col in final_df.columns}
    identity_columns = [col for col in final_df.columns if col not in described_by]

    final_df = (
        final_df
        .groupby(identity_columns, dropna=False, sort=False)
        .agg(described_by)
        .reset_index()
    )
    final_df = final_df[
        [col for col in FINAL_COLUMNS if col in final_df.columns] + ["operation_in_training_scope"]
    ]
else:
    final_df = final_df.drop_duplicates()

final_df = final_df.reset_index(drop=True)

print("Rows outside the trained operation set:", len(out_of_scope_rows),
      "(dropped)" if RESTRICT_TO_IN_SCOPE_OPERATIONS else "(kept, tagged operation_in_training_scope=False)")

if len(out_of_scope_rows):
    print(
        out_of_scope_rows["operation_specifier_final"]
        .astype(str).str.strip().str.lower().value_counts().to_string()
    )

print("Rows collapsed as material duplicates :", rows_before_material_collapse - len(final_df))
print("Final event rows                      :", len(final_df))

final_df.to_excel(FINAL_OUTPUT_PATH, index=False)
print("Final event table saved to:", FINAL_OUTPUT_PATH)

final_df.head(30)

## Cell 17 — Column check

In [ ]:
# Cell 17: a look at the long record table behind the final event table.

inspect_columns = [
    "article_id",
    "source_year",
    "operation_sentence_index",
    "parser_model_kind",
    "record_status",
    "operation_specifier",
    "operation_specifier_final",
    "operation_specifier_final_source",
    "vessel_text",
    "temperature_raw_value",
    "temperature_raw_units",
    "time_raw_value",
    "time_raw_units",
    "heating_rate_value",
    "heating_rate_units",
    "time_binding_rule",
    "time_binding_confidence",
    "time_model_binding_score",
    "temperature_rate_filter_status",
    "record_sentence",
]

combined_df[[col for col in inspect_columns if col in combined_df.columns]].head(20)